# 1. NaiveRAG 시스템 구현

- "고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf" 를 바탕으로 NaiveRAG를 구현하고 Langfuse를 사용하여 추적해봅니다.

## 환경설정

In [17]:
import os

# 1. macOS OpenMP 충돌 방지 (반드시 최상단 실행)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# 2. 토크나이저 병렬 처리 충돌 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("환경 설정 완료. 이제 다음 셀들을 실행하세요.")

환경 설정 완료. 이제 다음 셀들을 실행하세요.


In [18]:
from dotenv import load_dotenv

load_dotenv()

True

## RAG 기본 파이프라인(skeleton code)

In [19]:
# !pip install langchain==0.2.14 langchain-core==0.2.33 langchain-openai==0.1.22 langfuse==2.30.0

In [20]:
# 라이브러리 임포트

from langchain_text_splitters import RecursiveCharacterTextSplitter     # 분할기
from langchain_community.document_loaders import PyMuPDFLoader          # 로더
from langchain_community.vectorstores import FAISS                      # 벡터스토어
from langchain_core.output_parsers import StrOutputParser              
from langchain_core.runnables import RunnablePassthrough                
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse.callback import CallbackHandler

### 사전 준비단계

In [21]:
# 단계 1: 문서 로드(Load Documents)
loader = PyMuPDFLoader('/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf')
docs = loader.load()

len(docs)

297

In [22]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

In [23]:
len(split_documents)

431

In [24]:
print(split_documents[1].page_content)

-2- 
 
 
 
 
 
목   차
   
. 
Ⅰ사업개요
 ·················································································································· 
 4 
1. 
 
사업개요
 ·············································································································· 
 4 
2. 
 
사업배경
 ·············································································································· 
 4 
3. 
 
사업범위
 ·············································································································· 
 5 
4. 기대효과 ··············································································································· 
 7 
 
. 
Ⅱ현황 및 문제점
 ········································································································ 
 8 
1. 
 
일반현황
 ·············································································································· 
 8 
2. 
 
정보화현황


In [25]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x118cb1270>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x118cb13c0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [26]:
# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [27]:
# 단계 5: 검색기(Retreiever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x14bfb04c0>)

In [28]:
retriever.invoke("'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?",)

[Document(id='4d0f0ec4-377a-4933-894e-7d1f8d96e17b', metadata={'producer': 'LibreOffice 26.2.0.3 (AARCH64)', 'creator': 'Draw', 'creationdate': '2026-02-09T13:29:51+09:00', 'source': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'file_path': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'total_pages': 297, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260209132951+09'00'", 'page': 77}, page_content='-78- \n \n \n \n \n \n \n \n제공하여야함\n- 데이터베이스,  \n   \n프로그램\n \n❍ \n \n \n \n \n \n다국어콘텐츠탑재여부관리\n- \n  \n  \n  \n  \n  \n  \n  \n  \n  \n \n첨부문서나영상등의형태로포털에탑재되는콘텐츠는해당콘텐츠의\n \n \n \n \n영문버전탑재여부를시스템에서체크/\n \n \n \n \n관리할수있도록구현\n산출정보 \n• \n  \n프로그램목록,  프로그램명세서,  화면정의서,  \n  \n \n사용자매뉴얼등\n \n \n \n \n요구사항 분류 \n \n \n기능요구사항\n요구사항 번호 \nSFR-포털-009 \n요구사항 명칭 \n \n \n지능형검색\n요구 \n사항 \n상세 \n설명 \n정의 \n \n \n \n \n \n지능형검색적용방식정의\n세부 \n내용 \n❍ \n \n \n \n지능형검색방식\n- \n  \n 

In [34]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following poeces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Answer in Korean.

#Question:
{question}
#Context:
{context}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 를 생성합니다.
llm = ChatOpenAI(model_name="gpt-5-mini")

# 단계 8: 체인(Chain) 생성
langfuse_handler = CallbackHandler() 
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0.7 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}

received error response: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm that you've configured the correct host."}
received error response: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm that you've configured the correct host."}
received error response: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm that you've configured the correct host."}
Giving up execute_task_with_backoff(...) after 3 tries (langfuse.request.APIError: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm that you've configured the correct host."} (401): None)
Giving up execute_task_with_backoff(...) after 3 tries (langfuse.request.APIError: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm that you've configured the correct host."} (401): None)
Giving up execute_task_with_backoff(...) after 3 tries (langfuse.request.APIError: {'error': 'UnauthorizedError', 'message': "Invalid credentials. Confirm tha

In [ ]:
retrieved_chunks = retriever.invoke("'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num + 1) #실제 페이지는 +1

78
180
68
172


In [ ]:
question = "'성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?"
responce = chain.invoke(question)
print(responce)

학생이 성적을 조회하려면 담당교수가 해당 학생의 성적을 입력(처리)해 놓아야 하며, 그 조회는 성적처리일정(성적입력·성적조회 등 시작일자·종료일) 범위 내에 이루어져야 합니다.


In [ ]:
retrieved_chunks = retriever.invoke("'성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num + 1) #실제 페이지는 +1

109
109
106
107


#### 평가 데이터셋
Q. 이번 차세대 시스템 구축 사업에서 '지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식 3가지는 무엇인가요?	

A. 의미기반 검색(문맥 분석), 개인화/추천 검색(사용자 맞춤형 정보 제공), 유사문장 검색(유사도가 높은 최적 문장 제공) 등입니다. 

-78페이지-

Q. 차세대 학사행정시스템의 '성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?	

A.학생은 수강소감 설문응답을 완료해야만 성적 조회가 가능합니다. (단, 소속 학과에 따라 안전교육 이수 등 추가 조건이 있을 수 있음)

-109페이지-